# eBird API
## MAD 2502 — Applied Data

**Big idea:** An API lets one program request structured data from another service.

By the end, you should be able to explain an endpoint, header, parameter, and JSON response; turn API records into a DataFrame; distinguish rows from counts; and reuse one request for another species or region.

> **Lesson rhythm:** Understand → Predict → Run → Change → Use → Practice → Connect


## 1. Understand

We will use eBird's recent-observations endpoint:

```text
https://api.ebird.org/v2/data/obs/{region_code}/recent/{species_code}
```

Examples: Florida `US-FL`, Northern Cardinal `norcar`, Blue Jay `blujay`, American Crow `amecro`. The API key is sent in a request **header**; options such as the number of days are sent as **parameters**.

Keep your key in `my_api_info.py`:

```python
ebird_api_key = "PASTE-YOUR-KEY-HERE"
```

Never place a real key directly in a public notebook.


In [ ]:
import pandas as pd
import requests

try:
    import my_api_info
    API_KEY = my_api_info.ebird_api_key
    print("API key loaded.")
except (ImportError, AttributeError):
    API_KEY = None
    print("No API key found. Offline examples will still work.")


## 2. Predict

Before running the next cell, predict the exact URL produced by these values:

```python
region_code = "US-FL"
species_code = "norcar"
days = 7
```

Also predict: Is `days` part of the URL path, the header, or the parameter dictionary?


In [ ]:
region_code = "US-FL"
species_code = "norcar"
days = 7

url = (
    "https://api.ebird.org/v2/data/obs/"
    f"{region_code}/recent/{species_code}"
)
params = {"back": days, "maxResults": 5000}
headers = {"X-eBirdApiToken": API_KEY}

print(url)
print(params)


## 3. Run

Start with an offline response. A common JSON response becomes a Python **list of dictionaries**, which maps naturally to a DataFrame.


In [ ]:
practice_records = [
    {"speciesCode": "norcar", "locName": "Campus Garden", "howMany": 2},
    {"speciesCode": "norcar", "locName": "Lake Trail", "howMany": 5},
    {"speciesCode": "norcar", "locName": "Wetlands", "howMany": None},
]

practice_birds = pd.DataFrame(practice_records)
display(practice_birds)

counts = pd.to_numeric(practice_birds["howMany"], errors="coerce")
print("returned records:", len(practice_birds))
print("known bird count total:", counts.sum())
print("missing counts:", counts.isna().sum())


Compare the output with what you expected. `len(df)` counts returned records; `sum()` adds known bird counts. Those answer different questions. A missing `howMany` means unknown, not automatically zero.


## 4. Change

Change **one thing at a time** and predict the effect before running it.

- Change `species_code` from `norcar` to `blujay`. What part of the URL changes?
- Change `days` from `7` to `10`. What part of the request changes?
- Add one more pretend record with `howMany = 12`. What happens to `len(...)` and to the sum?


In [ ]:
species_code = "blujay"
days = 10

changed_url = (
    "https://api.ebird.org/v2/data/obs/"
    f"{region_code}/recent/{species_code}"
)
changed_params = {"back": days, "maxResults": 5000}

print(changed_url)
print(changed_params)


## 5. Use

Now turn the request into a reusable function. `raise_for_status()` is important: a failed request should stop instead of silently becoming fake good data.


In [ ]:
def get_ebird_data(api_key, region_code, species_code, days=7, max_results=5000):
    """Return recent eBird observations as a DataFrame."""
    endpoint = (
        "https://api.ebird.org/v2/data/obs/"
        f"{region_code}/recent/{species_code}"
    )
    request_headers = {"X-eBirdApiToken": api_key}
    request_params = {"back": days, "maxResults": max_results}

    response = requests.get(
        endpoint,
        headers=request_headers,
        params=request_params,
        timeout=30,
    )
    response.raise_for_status()

    data = pd.DataFrame(response.json())
    if "howMany" in data.columns:
        data["howMany"] = pd.to_numeric(data["howMany"], errors="coerce")
    return data


In [ ]:
if API_KEY:
    birds = get_ebird_data(API_KEY, "US-FL", "norcar", days=7)
    display(birds.head())
    print("records:", len(birds))
    if "howMany" in birds.columns:
        print("known count total:", birds["howMany"].sum())
else:
    print("Add your API key to run the live example.")


## 6. Practice

1. Retrieve Northern Cardinal (`norcar`) data for Florida for 7 days.
2. Count returned records.
3. Add the known `howMany` values.
4. Change to Blue Jay (`blujay`) and 10 days.
5. Change the region to another state.
6. Find the five locations appearing most often.
7. Explain why the sum of `howMany` is not a population estimate.


## 7. Connect

The same pattern appears in many APIs: build a request, receive structured data, convert it into useful Python objects, then ask a question.

For a capstone source, record: what one returned record represents, limits or caps, missing fields, authentication requirements, usage rules, and enough request information to reproduce the result later.

**Next connection:** pandas Basics focuses on what you can do after the data are already in a DataFrame.
